# H&M Customer Purchase Prediction — Kaggle End-to-End Run

Downloads the official H&M competition data with `kagglehub`, automatically finds the required CSV files, and runs the complete ML pipeline. **Images are not loaded or used.**

In [ ]:
# 1. Clone your GitHub repository
# Replace YOUR_GITHUB_REPO_URL with your actual repository URL.

!git clone YOUR_GITHUB_REPO_URL /kaggle/working/HM_Kaggle_Project
%cd /kaggle/working/HM_Kaggle_Project

In [ ]:
# 2. Install dependencies
!pip install -q kagglehub xgboost pyarrow joblib

In [ ]:
# 3. Download the official H&M competition data
import kagglehub

downloaded_path = kagglehub.competition_download(
    "h-and-m-personalized-fashion-recommendations"
)
print('kagglehub returned:', downloaded_path)

In [ ]:
# 4. Resolve the directory containing the three required CSV files
from pathlib import Path
import os

required = {'transactions_train.csv', 'customers.csv', 'articles.csv'}
dataset_dir = None
p = Path(downloaded_path)
roots = [p] if p.is_dir() else [p.parent]

for base in roots + [Path('/kaggle/input'), Path('/kaggle/working'), Path('/root/.cache')]:
    if not base.exists():
        continue
    try:
        for candidate in base.rglob('transactions_train.csv'):
            parent = candidate.parent
            names = {x.name for x in parent.iterdir()}
            if required.issubset(names):
                dataset_dir = parent
                break
    except (PermissionError, OSError):
        pass
    if dataset_dir is not None:
        break

if dataset_dir is None:
    raise FileNotFoundError('H&M competition files were not found after kagglehub download.')

os.environ['HM_DATA_DIR'] = str(dataset_dir)
print('H&M dataset directory:', dataset_dir)

In [ ]:
# 5. Verify the files
from pathlib import Path
data_dir = Path(os.environ['HM_DATA_DIR'])
for name in ['transactions_train.csv', 'customers.csv', 'articles.csv']:
    assert (data_dir / name).exists(), name
print('Dataset verification: PASSED')

In [ ]:
# 6. Run the complete leakage-safe training/evaluation pipeline
!python src/kaggle_runner.py

In [ ]:
# 7. Display model-wise metrics, baseline improvement, and final test metrics
import pandas as pd
from pathlib import Path

results = Path('results')
modelwise = pd.read_csv(results / 'final_model_comparison.csv')
improvement = pd.read_csv(results / 'baseline_improvement.csv')
best_test = pd.read_csv(results / 'best_model_test_metrics.csv')

print('MODEL-WISE VALIDATION METRICS')
display(modelwise[['model','val_accuracy','val_precision','val_recall','val_f1','val_roc_auc','val_pr_auc','val_opt_f1','fit_time_sec']])
print('\nBASELINE VS BEST MODEL IMPROVEMENT')
display(improvement)
print('\nBEST MODEL — FINAL TEST METRICS')
display(best_test)
print('\nBEST MODEL SUMMARY')
print((results / 'FINAL_SUMMARY.txt').read_text())